# CS 432 – Databases | Assignment 3 – Module A
## Transaction Management, Concurrency Control, and ACID Validation
**Instructor:** Dr. Yogesh K. Meena | Track 1 | Semester II (2025–2026)

---
## 1. Introduction

This notebook demonstrates the **complete transaction engine** built on top of the existing B+ Tree DBMS (Assignment 2).  
The implementation adds three new modules:

| Module | File | Responsibility |
|--------|------|---------------|
| Write-Ahead Log | `wal.py` | Persistent fsync'd log of every mutation |
| Transaction Manager | `transaction_manager.py` | begin / commit / rollback with READ COMMITTED isolation |
| Recovery Manager | `recovery.py` | ARIES-inspired 3-pass crash recovery |

### ACID Guarantees
- **Atomicity** – all ops in a transaction apply together or none do
- **Consistency** – DB + B+ Tree always match; PK constraints never violated
- **Isolation** – dirty reads blocked; each transaction sees only committed data
- **Durability** – COMMIT fsynced to WAL before returning; recovery rebuilds state


## 2. Setup

In [1]:
import sys, os, shutil, json
sys.path.insert(0, os.path.abspath('.'))

from database.db_manager import DatabaseManager
from database.transaction_manager import TransactionManager
from database.recovery import RecoveryManager
from database.wal import WALManager

WAL_DIR = "data/demo_wal"
os.makedirs(WAL_DIR, exist_ok=True)

def fresh_db(name="demo_db"):
    db = DatabaseManager(name)
    db.create_table("students",
        columns=["id","name","gpa","dept"], pk_column="id", order=4)
    return db

def fresh_tm(db, tag=""):
    p = f"{WAL_DIR}/wal_{tag}.log"
    if os.path.exists(p): os.remove(p)
    return TransactionManager(db, wal_path=p)

def make_student(i):
    return {"id":i, "name":f"Student_{i}", "gpa":round(7.0+(i%30)*0.1,1), "dept":"CS"}

print("All modules loaded successfully.")


All modules loaded successfully.


## 3. Atomicity

### 3.1 Successful Commit – all ops applied atomically


In [2]:
db = fresh_db("atom_db")
tm = fresh_tm(db, "atom")

t = tm.begin()
print(f"Transaction started: {t}")

for i in range(1, 6):
    tm.insert(t, "students", make_student(i))

# Before commit: DB must be completely untouched
before = [db.select("students", i) for i in range(1,6)]
print(f"DB before commit (all should be None): {before}")

tm.commit(t)

after = db.select_all("students")
print(f"DB after commit ({len(after)} rows):")
for r in after:
    print(f"  {r}")


Transaction started: T-73D99100
DB before commit (all should be None): [None, None, None, None, None]
DB after commit (5 rows):
  {'id': 1, 'name': 'Student_1', 'gpa': 7.1, 'dept': 'CS'}
  {'id': 2, 'name': 'Student_2', 'gpa': 7.2, 'dept': 'CS'}
  {'id': 3, 'name': 'Student_3', 'gpa': 7.3, 'dept': 'CS'}
  {'id': 4, 'name': 'Student_4', 'gpa': 7.4, 'dept': 'CS'}
  {'id': 5, 'name': 'Student_5', 'gpa': 7.5, 'dept': 'CS'}


### 3.2 Rollback – zero ops applied

In [3]:
db2 = fresh_db("rollback_db")
tm2 = fresh_tm(db2, "rollback")

t = tm2.begin()
tm2.insert(t, "students", make_student(100))
tm2.insert(t, "students", make_student(200))
print(f"Buffered 2 inserts in txn {t} (not yet applied)")

tm2.rollback(t)
print(f"Rolled back. Row count: {db2.count('students')} (expected 0)")
print(f"Row 100: {db2.select('students', 100)}  (expected None)")
print(f"Row 200: {db2.select('students', 200)}  (expected None)")


Buffered 2 inserts in txn T-361D28C6 (not yet applied)
Rolled back. Row count: 0 (expected 0)
Row 100: None  (expected None)
Row 200: None  (expected None)


### 3.3 Simulated Mid-Transaction Crash → Auto-Rollback

In [4]:
db3 = fresh_db("crash_db")
tm3 = fresh_tm(db3, "crash")

# Pre-commit row 5
t_pre = tm3.begin()
tm3.insert(t_pre, "students", make_student(5))
tm3.commit(t_pre)
print(f"Pre-committed row 5. Count: {db3.count('students')}")

# New txn tries to insert duplicate PK 5
t_bad = tm3.begin()
try:
    tm3.insert(t_bad, "students", make_student(5))   # should raise
except ValueError as e:
    print(f"Caught expected error: {e}")
tm3.rollback(t_bad)

print(f"After crash+rollback, count: {db3.count('students')} (expected 1)")


Pre-committed row 5. Count: 1
Caught expected error: Duplicate primary key 5 in 'students'
After crash+rollback, count: 1 (expected 1)


## 4. Consistency

### 4.1 DB and B+ Tree always in sync

In [5]:
db = fresh_db("cons_db")
tm = fresh_tm(db, "cons")

t = tm.begin()
for i in range(1, 9):
    tm.insert(t, "students", make_student(i))
tm.commit(t)

tbl = db.get_table("students")
db_rows   = db.select_all("students")
tree_rows = [v for _,v in tbl._index.get_all()]

print(f"DB rows  : {len(db_rows)}")
print(f"Tree rows: {len(tree_rows)}")
assert len(db_rows) == len(tree_rows), "MISMATCH!"
print("All IDs match:", [r['id'] for r in db_rows] == [r['id'] for r in tree_rows])

# Now rollback a txn and re-check
t2 = tm.begin()
tm.insert(t2, "students", make_student(999))
tm.rollback(t2)

db_rows2   = db.select_all("students")
tree_rows2 = [v for _,v in tbl._index.get_all()]
print(f"After rollback – DB: {len(db_rows2)}, Tree: {len(tree_rows2)} (both should be 8)")
assert 999 not in [r['id'] for r in tree_rows2], "Rolled-back row leaked into B+ Tree!"
print("Consistency check PASSED")


DB rows  : 8
Tree rows: 8
All IDs match: True
After rollback – DB: 8, Tree: 8 (both should be 8)
Consistency check PASSED


## 5. Isolation (READ COMMITTED)

### 5.1 Dirty Read Prevention

In [6]:
db = fresh_db("iso_db")
tm = fresh_tm(db, "iso")

t_a = tm.begin()
tm.insert(t_a, "students", make_student(42))
print(f"T_A inserted row 42 but NOT committed yet")

t_b = tm.begin()
row = tm.select(t_b, "students", 42)
print(f"T_B reads row 42 (should be None – no dirty read): {row}")

tm.commit(t_a)
row_after = db.select("students", 42)
print(f"After T_A commits, committed read of row 42: {row_after}")
tm.rollback(t_b)


T_A inserted row 42 but NOT committed yet
T_B reads row 42 (should be None – no dirty read): None
After T_A commits, committed read of row 42: {'id': 42, 'name': 'Student_42', 'gpa': 8.2, 'dept': 'CS'}


### 5.2 Two Concurrent Transactions – No Cross-Visibility of Pending Writes

In [7]:
import threading

db = fresh_db("iso2_db")
tm = fresh_tm(db, "iso2")
results = {}

def txn_a():
    t = tm.begin()
    tm.insert(t, "students", make_student(1))
    # Read T_B's row before T_B commits
    results['a_sees_2_before_commit'] = tm.select(t, "students", 2)
    tm.commit(t)

def txn_b():
    t = tm.begin()
    tm.insert(t, "students", make_student(2))
    results['b_sees_1_before_commit'] = tm.select(t, "students", 1)
    tm.commit(t)

# Run sequentially (deterministic isolation demo)
txn_a()
txn_b()

print(f"T_A saw T_B row 2 before T_B committed: {results['a_sees_2_before_commit']} (expected None)")
print(f"T_B saw T_A row 1 before T_A committed: {results['b_sees_1_before_commit']}")
print(f"Final committed rows: {db.count('students')}")


T_A saw T_B row 2 before T_B committed: None (expected None)
T_B saw T_A row 1 before T_A committed: {'id': 1, 'name': 'Student_1', 'gpa': 7.1, 'dept': 'CS'}
Final committed rows: 2


## 6. Durability

### 6.1 WAL Contents After Commit

In [8]:
db = fresh_db("dur_db")
wal_path = f"{WAL_DIR}/wal_dur.log"
if os.path.exists(wal_path): os.remove(wal_path)
tm = TransactionManager(db, wal_path=wal_path)

t = tm.begin()
tm.insert(t, "students", make_student(7))
tm.insert(t, "students", make_student(8))
tm.commit(t)

records = tm.wal.read_all()
print(f"WAL records written ({len(records)} total):")
for r in records:
    print(f"  LSN={r.lsn:02d}  op={r.op:<8}  txn={r.txn_id}  key={r.key}  ts={r.ts}")


WAL records written (4 total):
  LSN=01  op=BEGIN     txn=T-6F37D5A6  key=None  ts=2026-04-05T16:24:35
  LSN=02  op=INSERT    txn=T-6F37D5A6  key=7  ts=2026-04-05T16:24:35
  LSN=03  op=INSERT    txn=T-6F37D5A6  key=8  ts=2026-04-05T16:24:35
  LSN=04  op=COMMIT    txn=T-6F37D5A6  key=None  ts=2026-04-05T16:24:35


## 7. Crash Recovery

### 7.1 Three-Pass ARIES Recovery

In [9]:
from database.wal import WALManager

wal_path = f"{WAL_DIR}/wal_recovery.log"
if os.path.exists(wal_path): os.remove(wal_path)

# ── Session 1: commit T_ok, then "crash" mid T_crash ─────────────────────────
db1 = fresh_db("recovery_s1")
tm1 = TransactionManager(db1, wal_path=wal_path)

t_ok = tm1.begin()
for i in range(1, 4):
    tm1.insert(t_ok, "students", make_student(i))
tm1.commit(t_ok)
print(f"Session 1: committed rows 1-3")

# Simulate crash: write BEGIN+INSERT to WAL but never COMMIT
wal_raw = WALManager(log_path=wal_path)
wal_raw.append("T-CRASH", "BEGIN")
wal_raw.append("T-CRASH", "INSERT", "students", 99, before=None, after=make_student(99))
print("Session 1: crashed mid-transaction T-CRASH (no COMMIT written)")

# ── Session 2: start fresh, run recovery ─────────────────────────────────────
db2 = fresh_db("recovery_s2")
rm  = RecoveryManager(db2, wal_path=wal_path)
report = rm.recover()

print("\n" + report.summary())
print(f"\nRow 1 (committed): {db2.select('students', 1)}")
print(f"Row 99 (crashed) : {db2.select('students', 99)}")
print(f"Total rows after recovery: {db2.count('students')} (expected 3)")


Session 1: committed rows 1-3
Session 1: crashed mid-transaction T-CRASH (no COMMIT written)

═══════════════════════════════════════════════════════
  CRASH RECOVERY REPORT
═══════════════════════════════════════════════════════
  Committed transactions  : 1  ['T-1B6F2646']
  Aborted  transactions   : 0  []
  Loser    transactions   : 1  ['T-CRASH']
  Redo operations applied : 3
  Undo operations applied : 1
  Status : CLEAN – database is consistent
═══════════════════════════════════════════════════════

Row 1 (committed): {'id': 1, 'name': 'Student_1', 'gpa': 7.1, 'dept': 'CS'}
Row 99 (crashed) : None
Total rows after recovery: 3 (expected 3)


## 8. Full ACID Test Suite (15 Tests)

In [10]:
import unittest, shutil

# Clean up any leftover WAL files
test_wal_dir = "data/test_wal"
if os.path.exists(test_wal_dir):
    shutil.rmtree(test_wal_dir)
os.makedirs(test_wal_dir, exist_ok=True)

from tests.test_acid import TestACID

loader = unittest.TestLoader()
suite  = loader.loadTestsFromTestCase(TestACID)

import io
buf = io.StringIO()
runner = unittest.TextTestRunner(stream=buf, verbosity=2)
result = runner.run(suite)

print(buf.getvalue())
print(f"{'='*55}")
print(f"  Ran: {result.testsRun}  |  Passed: {result.testsRun - len(result.failures) - len(result.errors)}  |  Failed: {len(result.failures)}  |  Errors: {len(result.errors)}")
print(f"  Status: {'ALL PASSED ✅' if result.wasSuccessful() else 'SOME FAILED ❌'}")
print(f"{'='*55}")


test_T01_atomicity_commit_applies_all (tests.test_acid.TestACID.test_T01_atomicity_commit_applies_all) ... ok
test_T02_atomicity_rollback_applies_none (tests.test_acid.TestACID.test_T02_atomicity_rollback_applies_none) ... ok
test_T03_atomicity_exception_during_commit (tests.test_acid.TestACID.test_T03_atomicity_exception_during_commit) ... ok
test_T04_consistency_pk_uniqueness (tests.test_acid.TestACID.test_T04_consistency_pk_uniqueness) ... ok
test_T05_consistency_db_bptree_sync_after_commit (tests.test_acid.TestACID.test_T05_consistency_db_bptree_sync_after_commit) ... ok
test_T06_consistency_db_bptree_sync_after_rollback (tests.test_acid.TestACID.test_T06_consistency_db_bptree_sync_after_rollback) ... ok
test_T07_isolation_no_dirty_read (tests.test_acid.TestACID.test_T07_isolation_no_dirty_read) ... ok
test_T08_isolation_concurrent_txns (tests.test_acid.TestACID.test_T08_isolation_concurrent_txns) ... ok
test_T09_durability_commit_record_in_wal (tests.test_acid.TestACID.test_T09_du

## 9. Conclusion

### What was implemented

| Component | File | Key Feature |
|-----------|------|-------------|
| Write-Ahead Log | `wal.py` | JSON-line log, fsync on every write, LSN-ordered |
| Transaction Manager | `transaction_manager.py` | begin/commit/rollback, READ COMMITTED isolation, dirty-read prevention |
| Recovery Manager | `recovery.py` | 3-pass ARIES: Analysis → Redo → Undo |
| ACID Test Suite | `tests/test_acid.py` | 15 tests covering all 4 ACID properties |

### ACID verification summary

| Property | Tests | Technique |
|----------|-------|-----------|
| Atomicity | T01, T02, T03, T13 | Ops buffered in memory; applied only on commit |
| Consistency | T04, T05, T06 | PK uniqueness checks; DB + B+ Tree always compared |
| Isolation | T07, T08, T14 | Dirty reads blocked by read-committed snapshot |
| Durability | T09, T10, T11, T12, T15 | WAL fsynced before commit returns; recovery rebuilds state |

### Design choices
- **WAL before B+ Tree**: every mutation is logged *before* touching the index → no committed data lost on crash
- **Before-images in WAL**: every UPDATE and DELETE stores the pre-image → enables undo without a separate undo log
- **Idempotent redo**: recovery skips inserts/updates whose key already has the target value → safe to run multiple times
- **Thread-safe WAL**: `threading.Lock` guards the append path → correct under concurrent transactions
